In [1]:
import pandas as pd

In [92]:
## PADI
padi = pd.read_excel("/global/scratch/projects/fc_mvslab/OpenProjects/Sanjana/collaborations/GrENEnet_TF_hits/08b_padi_ADs.xlsx")
padi = padi[["Fragment Sequence", "PADI Score"]]
padi = padi.rename(columns = {"Fragment Sequence" : "seq", "PADI Score": "score"})
padi["screen"] = "padi"
padi["seq"] = padi["seq"].str.strip().str.upper()
padi = padi.drop_duplicates(subset = "seq")
padi = padi.dropna()
padi["active"] = padi["score"] > 1
padi

,seq,score,screen,active
0,MEDQVGFGFRPNDEELVGHYLRNKIEGNTSRDVEVAISEV,1.004699,padi,True
1,PNDEELVGHYLRNKIEGNTSRDVEVAISEVNICSYDPWNL,0.575876,padi,False
2,LRNKIEGNTSRDVEVAISEVNICSYDPWNLRFQSKYKSRD,-0.300407,padi,False
3,RDVEVAISEVNICSYDPWNLRFQSKYKSRDAMWYFFSRRE,0.105372,padi,False
4,NICSYDPWNLRFQSKYKSRDAMWYFFSRRENNKGNRQSRT,-0.167147,padi,False
...,...,...,...,...
69568,SDFVVKAEDVEASMAIFWTPGTRVKMAMETEDSSRITWFQ,-0.139281,padi,False
69569,EASMAIFWTPGTRVKMAMETEDSSRITWFQGIVFYTYQET,-0.064551,padi,False
69570,GTRVKMAMETEDSSRITWFQGIVFYTYQETGPWRGSPWNS,-0.410777,padi,False
69571,EDSSRITWFQGIVFYTYQETGPWRGSPWNSFRYKIPFTIP,0.329820,padi,False


In [93]:
padi["active"].value_counts()

active
False    62262
True      6196
Name: count, dtype: int64

In [94]:
seqs = pd.read_csv("01d_tair_arabidopsis_cleaned.csv", index_col = 0)
seqs

,id,seq,gene,tair_ID,seq_cleaned
0,AT5G03200.1 | Symbols: LUL1 | LOG2-LIKE UBIQUI...,MGNLISLIFCCGRRQRSNIPPAMETAPLELPPNRFVFAAVPPYLNP...,LUL1,AT5G03200.1,MGNLISLIFCCGRRQRSNIPPAMETAPLELPPNRFVFAAVPPYLNP...
1,"AT5G44110.1 | Symbols: ATNAP2, POP1, ABCI21, A...",MAEKNASAVDGAIRVSGMQFSYDVQDPIFFDFNLDLPAGSRCLLVG...,"ATNAP2, POP1, ABCI21, ATPOP1",AT5G44110.1,MAEKNASAVDGAIRVSGMQFSYDVQDPIFFDFNLDLPAGSRCLLVG...
2,"AT2G27030.3 | Symbols: ACAM-2, CAM5 | calmodul...",MADQLTDDQISEFKEAFSLFDKDGDGCITTKELGTVMRSLGQNPTE...,"ACAM-2, CAM5",AT2G27030.3,MADQLTDDQISEFKEAFSLFDKDGDGCITTKELGTVMRSLGQNPTE...
3,AT1G23550.1 | Symbols: SRO2 | similar to RCD o...,MAAQVEIEDQTSVTNLDNGEIFDSISDDADSSVSHAGSSFSSSSLI...,SRO2,AT1G23550.1,MAAQVEIEDQTSVTNLDNGEIFDSISDDADSSVSHAGSSFSSSSLI...
4,"AT4G34240.1 | Symbols: ALDH3I1, ALDH3 | aldehy...",MTKLLEINHIQTLCFAKGFSPARLNVATSPFRISRRGGGGYCSNAC...,"ALDH3I1, ALDH3",AT4G34240.1,MTKLLEINHIQTLCFAKGFSPARLNVATSPFRISRRGGGGYCSNAC...
...,...,...,...,...,...
114,AT1G55320.1 | Symbols: AAE18 | acyl-activating...,MWKSIGELSCDDYVKAGLTLEDAKEFDKLVSDVITKAIETDPRDQW...,AAE18,AT1G55320.1,MWKSIGELSCDDYVKAGLTLEDAKEFDKLVSDVITKAIETDPRDQW...
115,"AT2G01570.1 | Symbols: RGA, RGA1, RGA24 | REPR...",MKRDHHQFQGRLSNHGTSSSSSSISKDKMMMVKKEEDGGGNMDDEL...,"RGA, RGA1, RGA24",AT2G01570.1,MKRDHHQFQGRLSNHGTSSSSSSISKDKMMMVKKEEDGGGNMDDEL...
116,"AT1G15040.1 | Symbols: GAT, GAT1_2.1 | glutami...",MVVANDLSSKILPRVLIVSRRTLRKNKYVDFVGEYHLDLIVSSGAV...,"GAT, GAT1_2.1",AT1G15040.1,MVVANDLSSKILPRVLIVSRRTLRKNKYVDFVGEYHLDLIVSSGAV...
117,AT1G14980.1 | Symbols: CPN10 | chaperonin 10 |...,MMKRLIPTFNRILVQRVIQPAKTESGILLPEKSSKLNSGKVIAVGP...,CPN10,AT1G14980.1,MMKRLIPTFNRILVQRVIQPAKTESGILLPEKSSKLNSGKVIAVGP...


In [95]:
import ahocorasick

A = ahocorasick.Automaton()

for i, row in padi.iterrows():
    seq = row["seq"]
    A.add_word(seq, (i, seq))  # store index + sequence

A.make_automaton()

In [154]:
matches = []

for _, prot in seqs.iterrows():
    prot_id = prot["tair_ID"]
    prot_seq = prot["seq_cleaned"]
    
    for end_idx, (tile_idx, tile_seq) in A.iter(prot_seq):
        start_idx = end_idx - len(tile_seq) + 1
        
        matches.append({
            "gene" : prot['gene'],
            "tair_ID": prot_id,
            "tile_index": tile_idx,
            "tile_seq": tile_seq,
            "start": start_idx,
            "end": end_idx,
        })

In [155]:
import pandas as pd

matches_df = pd.DataFrame(matches)

# bring in score / activity
matches_df = matches_df.merge(
    padi.reset_index().rename(columns={"index": "tile_index"}),
    on="tile_index",
    how="left"
)
matches_df

,gene,tair_ID,tile_index,tile_seq,start,end,seq,score,screen,active
0,"ERF109, RRTF1",AT4G34410.1,50418,MHYPNNRTEFVGAPAPTRYQKEQLSPEQELSVIVSALQHV,0,39,MHYPNNRTEFVGAPAPTRYQKEQLSPEQELSVIVSALQHV,-0.510879,padi,False
1,"ERF109, RRTF1",AT4G34410.1,50419,VGAPAPTRYQKEQLSPEQELSVIVSALQHVISGENETAPC,10,49,VGAPAPTRYQKEQLSPEQELSVIVSALQHVISGENETAPC,-0.543296,padi,False
2,"ERF109, RRTF1",AT4G34410.1,50420,KEQLSPEQELSVIVSALQHVISGENETAPCQGFSSDSTVI,20,59,KEQLSPEQELSVIVSALQHVISGENETAPCQGFSSDSTVI,-0.532311,padi,False
3,"ERF109, RRTF1",AT4G34410.1,50421,SVIVSALQHVISGENETAPCQGFSSDSTVISAGMPRLDSD,30,69,SVIVSALQHVISGENETAPCQGFSSDSTVISAGMPRLDSD,-0.291246,padi,False
4,"ERF109, RRTF1",AT4G34410.1,50422,ISGENETAPCQGFSSDSTVISAGMPRLDSDTCQVCRIEGC,40,79,ISGENETAPCQGFSSDSTVISAGMPRLDSDTCQVCRIEGC,1.583069,padi,True
...,...,...,...,...,...,...,...,...,...,...
534,"RGA, RGA1, RGA24",AT2G01570.1,18466,RVERHETLSQWGNRFGSSGLAPAHLGSNAFKQASMLLSVF,510,549,RVERHETLSQWGNRFGSSGLAPAHLGSNAFKQASMLLSVF,-0.023732,padi,False
535,"RGA, RGA1, RGA24",AT2G01570.1,18467,WGNRFGSSGLAPAHLGSNAFKQASMLLSVFNSGQGYRVEE,520,559,WGNRFGSSGLAPAHLGSNAFKQASMLLSVFNSGQGYRVEE,-0.027069,padi,False
536,"RGA, RGA1, RGA24",AT2G01570.1,18468,APAHLGSNAFKQASMLLSVFNSGQGYRVEESNGCLMLGWH,530,569,APAHLGSNAFKQASMLLSVFNSGQGYRVEESNGCLMLGWH,-0.120358,padi,False
537,"RGA, RGA1, RGA24",AT2G01570.1,18469,KQASMLLSVFNSGQGYRVEESNGCLMLGWHTRPLITTSAW,540,579,KQASMLLSVFNSGQGYRVEESNGCLMLGWHTRPLITTSAW,-0.242184,padi,False


In [156]:
matches_df["tair_ID"].value_counts()

tair_ID
AT1G04050.1    70
AT2G01570.1    56
AT1G14920.1    51
AT4G31610.2    46
AT4G13980.1    44
AT4G18880.1    38
AT1G45249.6    35
AT5G22290.1    32
AT4G34530.1    31
AT3G15500.1    29
AT4G34410.1    24
AT1G04100.1    24
AT5G25160.1    22
AT5G47670.1    21
AT5G25190.1    16
Name: count, dtype: int64

In [157]:
# There are 15 TFs with activities measured in ARF!
len(matches_df["tair_ID"].value_counts())

15

In [158]:
matches_df["active"].value_counts()

active
False    474
True      65
Name: count, dtype: int64

In [159]:
matches_df["tile_center"] = (matches_df["start"] + matches_df["end"]) / 2
matches_df

,gene,tair_ID,tile_index,tile_seq,start,end,seq,score,screen,active,tile_center
0,"ERF109, RRTF1",AT4G34410.1,50418,MHYPNNRTEFVGAPAPTRYQKEQLSPEQELSVIVSALQHV,0,39,MHYPNNRTEFVGAPAPTRYQKEQLSPEQELSVIVSALQHV,-0.510879,padi,False,19.5
1,"ERF109, RRTF1",AT4G34410.1,50419,VGAPAPTRYQKEQLSPEQELSVIVSALQHVISGENETAPC,10,49,VGAPAPTRYQKEQLSPEQELSVIVSALQHVISGENETAPC,-0.543296,padi,False,29.5
2,"ERF109, RRTF1",AT4G34410.1,50420,KEQLSPEQELSVIVSALQHVISGENETAPCQGFSSDSTVI,20,59,KEQLSPEQELSVIVSALQHVISGENETAPCQGFSSDSTVI,-0.532311,padi,False,39.5
3,"ERF109, RRTF1",AT4G34410.1,50421,SVIVSALQHVISGENETAPCQGFSSDSTVISAGMPRLDSD,30,69,SVIVSALQHVISGENETAPCQGFSSDSTVISAGMPRLDSD,-0.291246,padi,False,49.5
4,"ERF109, RRTF1",AT4G34410.1,50422,ISGENETAPCQGFSSDSTVISAGMPRLDSDTCQVCRIEGC,40,79,ISGENETAPCQGFSSDSTVISAGMPRLDSDTCQVCRIEGC,1.583069,padi,True,59.5
...,...,...,...,...,...,...,...,...,...,...,...
534,"RGA, RGA1, RGA24",AT2G01570.1,18466,RVERHETLSQWGNRFGSSGLAPAHLGSNAFKQASMLLSVF,510,549,RVERHETLSQWGNRFGSSGLAPAHLGSNAFKQASMLLSVF,-0.023732,padi,False,529.5
535,"RGA, RGA1, RGA24",AT2G01570.1,18467,WGNRFGSSGLAPAHLGSNAFKQASMLLSVFNSGQGYRVEE,520,559,WGNRFGSSGLAPAHLGSNAFKQASMLLSVFNSGQGYRVEE,-0.027069,padi,False,539.5
536,"RGA, RGA1, RGA24",AT2G01570.1,18468,APAHLGSNAFKQASMLLSVFNSGQGYRVEESNGCLMLGWH,530,569,APAHLGSNAFKQASMLLSVFNSGQGYRVEESNGCLMLGWH,-0.120358,padi,False,549.5
537,"RGA, RGA1, RGA24",AT2G01570.1,18469,KQASMLLSVFNSGQGYRVEESNGCLMLGWHTRPLITTSAW,540,579,KQASMLLSVFNSGQGYRVEESNGCLMLGWHTRPLITTSAW,-0.242184,padi,False,559.5


In [179]:
matches_df[["gene", "tair_ID", "tile_center", "score"]].to_csv("10_padi_activity_scores.csv")

In [166]:
for index in matches_df[["gene", "tair_ID"]].drop_duplicates().iterrows():
    print(index)

(0, gene       ERF109, RRTF1
tair_ID      AT4G34410.1
Name: 0, dtype: object)
(24, gene              CIB1
tair_ID    AT4G34530.1
Name: 24, dtype: object)
(55, gene       ATREM1, REM1, REM34
tair_ID            AT4G31610.2
Name: 55, dtype: object)
(101, gene       AtABF2, ABF2, ATAREB1, AREB1
tair_ID                     AT1G45249.6
Name: 101, dtype: object)
(136, gene       L1L, NF-YB6
tair_ID    AT5G47670.1
Name: 136, dtype: object)
(157, gene       AT-HSFA5, HSFA5
tair_ID        AT4G13980.1
Name: 157, dtype: object)
(201, gene       NTL14, NAC089, FSQ6, anac089
tair_ID                     AT5G22290.1
Name: 201, dtype: object)
(233, gene              ESE3
tair_ID    AT5G25190.1
Name: 233, dtype: object)
(249, gene       ANAC55, ATNAC3, NAC055, ANAC055, NAC3
tair_ID                              AT3G15500.1
Name: 249, dtype: object)
(278, gene       SUVR1, SDG13
tair_ID     AT1G04050.1
Name: 278, dtype: object)
(348, gene             IAA10
tair_ID    AT1G04100.1
Name: 348, dtype: object)


In [170]:
import os

for _, row in matches_df[["gene", "tair_ID"]].drop_duplicates().iterrows():
    tair_ID, gene = row["tair_ID"], row["gene"]
    
    tf_tiles = merged[merged["uniprotID"] == tair_ID].sort_values("tile_center")
    
    fig, ax = plt.subplots(figsize=(7,3), dpi=300)
    ax.plot(tf_tiles["tile_center"], tf_tiles["activity"], label="Raw tile activity", marker='o', markersize=5)
    
    sns.despine()
    plt.title(f"{gene} | {tair_ID}")
    
    output_folder = "/global/scratch/projects/fc_mvslab/OpenProjects/Sanjana/collaborations/GrENEnet_TF_hits/09_measured_activity_traces"
    safe_name = f"{gene} | {tair_ID}"
    save_path = os.path.join(output_folder, safe_name + ".png")
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()

In [171]:
import pandas as pd

def get_active_intervals(df):
    active_intervals = []

    for tair_id, sub in df[df["active"]].groupby("tair_ID"):
        # sort by start position
        sub = sub.sort_values("start")

        current_start = None
        current_end = None

        for _, row in sub.iterrows():
            s, e = row["start"], row["end"]

            if current_start is None:
                # initialize first interval
                current_start, current_end = s, e
            elif s <= current_end:  
                # overlap or directly adjacent → extend interval
                current_end = max(current_end, e)
            else:
                # gap → save previous interval
                active_intervals.append({
                    "tair_ID": tair_id,
                    "start": current_start,
                    "end": current_end
                })
                current_start, current_end = s, e

        # save last interval
        if current_start is not None:
            active_intervals.append({
                "tair_ID": tair_id,
                "start": current_start,
                "end": current_end
            })

    return pd.DataFrame(active_intervals)

In [173]:
arf_ADs = get_active_intervals(matches_df)
arf_ADs

,tair_ID,start,end
0,AT1G04050.1,10,79
1,AT1G04050.1,610,649
2,AT1G04050.1,650,719
3,AT1G04100.1,170,239
4,AT1G14920.1,20,59
5,AT1G14920.1,60,119
6,AT1G14920.1,330,379
7,AT1G14920.1,380,439
8,AT1G45249.6,15,94
9,AT1G45249.6,205,244


In [174]:
arf_ADs.to_csv("10_padi_annotated_ADs.csv")

In [176]:
len(arf_ADs["tair_ID"].value_counts())

11

In [177]:
# 11 TFs with annotated ADs

arf_ADs["tair_ID"].value_counts()

tair_ID
AT1G14920.1    4
AT1G04050.1    3
AT5G22290.1    3
AT4G18880.1    3
AT4G34410.1    2
AT1G45249.6    2
AT2G01570.1    2
AT4G34530.1    2
AT3G15500.1    2
AT1G04100.1    1
AT4G13980.1    1
Name: count, dtype: int64